In [0]:
-- ============================================================================
-- MegaMart Databricks Capstone
-- Stage: 04 - Sales Analysis
--
-- Purpose:
--   Analyze cleaned Silver sales data and create dashboard-ready Gold
--   datasets for store and product sales performance.
--
-- Source:
--   megamart_dev.silver.*
--
-- Target:
--   megamart_dev.gold.*
--
-- Important:
--   The data-generation script is the source of truth for the actual
--   transaction schema. Sales transactions contain:
--       transaction_id
--       store_id
--       product_id
--       quantity_sold
--       unit_price
--       total_amount
--       sale_date
--
--   True profitability is NOT calculated because the generated source
--   contains selling price but no product unit cost.
-- ============================================================================


-- ============================================================================
-- 1. Select Project Catalog
-- ============================================================================

USE CATALOG megamart_dev;


-- ============================================================================
-- 2. Verify Silver Source Counts
-- ============================================================================

SELECT
    'sales_transactions' AS dataset,
    COUNT(*) AS record_count
FROM silver.sales_transactions

UNION ALL

SELECT
    'stores' AS dataset,
    COUNT(*) AS record_count
FROM silver.stores

UNION ALL

SELECT
    'products' AS dataset,
    COUNT(*) AS record_count
FROM silver.products;


-- ============================================================================
-- 3. Store Sales Summary
--
-- Answers:
--   - Total revenue by store
--   - Transaction count by store
--   - Average transaction value
--   - Total quantity sold
--   - Unique products sold
-- ============================================================================

CREATE OR REPLACE TABLE gold.store_sales_summary
USING DELTA
AS
SELECT
    s.store_id,
    s.store_name,
    s.location,

    COUNT(DISTINCT st.transaction_id) AS transaction_count,

    SUM(st.quantity_sold) AS total_quantity_sold,

    SUM(st.total_amount) AS total_revenue,

    ROUND(
        SUM(st.total_amount)
        / NULLIF(COUNT(DISTINCT st.transaction_id), 0),
        2
    ) AS average_transaction_value,

    COUNT(DISTINCT st.product_id) AS unique_products_sold

FROM silver.sales_transactions st

INNER JOIN silver.stores s
    ON st.store_id = s.store_id

GROUP BY
    s.store_id,
    s.store_name,
    s.location;


-- ============================================================================
-- 4. Top 5 Best-Selling Products
--
-- Primary ranking criterion:
--   Total quantity sold
--
-- Revenue is included as supporting information.
-- ============================================================================

CREATE OR REPLACE TABLE gold.top_products
USING DELTA
AS
WITH product_sales AS (

    SELECT
        st.product_id,
        p.product_name,
        p.category,

        SUM(st.quantity_sold) AS total_quantity_sold,

        SUM(st.total_amount) AS total_revenue,

        COUNT(DISTINCT st.transaction_id) AS transaction_count

    FROM silver.sales_transactions st

    INNER JOIN silver.products p
        ON st.product_id = p.product_id

    GROUP BY
        st.product_id,
        p.product_name,
        p.category
),

ranked_products AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY
                total_quantity_sold DESC,
                total_revenue DESC,
                product_id
        ) AS quantity_rank

    FROM product_sales
)

SELECT
    quantity_rank,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue,
    transaction_count

FROM ranked_products

WHERE quantity_rank <= 5;


-- ============================================================================
-- 5. Store Revenue and Transaction Ranking
--
-- Produces two independent rankings:
--   revenue_rank
--   transaction_count_rank
--
-- No single "overall" ranking is created.
-- ============================================================================

CREATE OR REPLACE TABLE gold.store_revenue_ranking
USING DELTA
AS
SELECT
    store_id,
    store_name,
    location,

    total_revenue,
    transaction_count,

    DENSE_RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank,

    DENSE_RANK() OVER (
        ORDER BY transaction_count DESC
    ) AS transaction_count_rank

FROM gold.store_sales_summary;


-- ============================================================================
-- 6. Monthly Sales by Store
--
-- Creates a complete Jan-Dec 2024 monthly grid for each store.
--
-- Metrics:
--   monthly revenue
--   transaction count
--   quantity sold
--   previous month revenue
--   MoM revenue change
--   MoM revenue growth %
-- ============================================================================

CREATE OR REPLACE TABLE gold.monthly_store_sales
USING DELTA
AS
WITH months AS (

    SELECT
        ADD_MONTHS(
            DATE '2024-01-01',
            month_number
        ) AS month_start

    FROM (
        SELECT EXPLODE(
            SEQUENCE(0, 11)
        ) AS month_number
    )
),

store_months AS (

    SELECT
        s.store_id,
        s.store_name,
        s.location,
        m.month_start

    FROM silver.stores s
    CROSS JOIN months m
),

monthly_sales AS (

    SELECT
        store_id,

        DATE_TRUNC(
            'MONTH',
            sale_date
        ) AS month_start,

        SUM(quantity_sold) AS monthly_quantity_sold,

        SUM(total_amount) AS monthly_revenue,

        COUNT(DISTINCT transaction_id)
            AS monthly_transaction_count

    FROM silver.sales_transactions

    GROUP BY
        store_id,
        DATE_TRUNC(
            'MONTH',
            sale_date
        )
),

combined AS (

    SELECT
        sm.store_id,
        sm.store_name,
        sm.location,
        sm.month_start,

        COALESCE(
            ms.monthly_quantity_sold,
            0
        ) AS monthly_quantity_sold,

        COALESCE(
            ms.monthly_revenue,
            0
        ) AS monthly_revenue,

        COALESCE(
            ms.monthly_transaction_count,
            0
        ) AS monthly_transaction_count

    FROM store_months sm

    LEFT JOIN monthly_sales ms
        ON sm.store_id = ms.store_id
        AND sm.month_start = ms.month_start
),

with_previous_month AS (

    SELECT
        *,

        LAG(monthly_revenue) OVER (
            PARTITION BY store_id
            ORDER BY month_start
        ) AS previous_month_revenue

    FROM combined
)

SELECT
    store_id,
    store_name,
    location,
    month_start,
    MONTH(month_start) AS month_number,

    DATE_FORMAT(
        month_start,
        'yyyy-MM'
    ) AS year_month,

    monthly_quantity_sold,
    monthly_transaction_count,
    monthly_revenue,

    previous_month_revenue,

    ROUND(
        monthly_revenue
        - COALESCE(previous_month_revenue, 0),
        2
    ) AS mom_revenue_change,

    CASE
        WHEN previous_month_revenue IS NULL THEN NULL

        WHEN previous_month_revenue = 0
            AND monthly_revenue > 0
            THEN NULL

        ELSE ROUND(
            (
                monthly_revenue
                - previous_month_revenue
            )
            / previous_month_revenue
            * 100,
            2
        )
    END AS mom_revenue_growth_pct

FROM with_previous_month;


-- ============================================================================
-- 7. Product Sales Performance by Store
--
-- Answers:
--   - Which products perform strongly within each store?
--   - What is each product's quantity and revenue contribution?
--
-- Important:
--   This is NOT a profitability table.
--   The source generator does not provide unit cost.
--   Therefore revenue contribution is used instead of fabricated profit.
-- ============================================================================

CREATE OR REPLACE TABLE gold.product_store_sales
USING DELTA
AS
WITH store_product_sales AS (

    SELECT
        st.store_id,
        s.store_name,

        st.product_id,
        p.product_name,
        p.category,

        SUM(st.quantity_sold) AS total_quantity_sold,

        SUM(st.total_amount) AS total_revenue,

        COUNT(DISTINCT st.transaction_id)
            AS transaction_count,

        AVG(st.unit_price) AS average_selling_price

    FROM silver.sales_transactions st

    INNER JOIN silver.stores s
        ON st.store_id = s.store_id

    INNER JOIN silver.products p
        ON st.product_id = p.product_id

    GROUP BY
        st.store_id,
        s.store_name,
        st.product_id,
        p.product_name,
        p.category
),

store_totals AS (

    SELECT
        store_id,
        SUM(total_revenue) AS store_total_revenue
    FROM store_product_sales
    GROUP BY store_id
)

SELECT
    sps.store_id,
    sps.store_name,

    sps.product_id,
    sps.product_name,
    sps.category,

    sps.total_quantity_sold,
    sps.transaction_count,
    ROUND(
        sps.average_selling_price,
        2
    ) AS average_selling_price,

    ROUND(
        sps.total_revenue,
        2
    ) AS total_revenue,

    ROUND(
        (
            sps.total_revenue
            / NULLIF(st.store_total_revenue, 0)
        ) * 100,
        2
    ) AS store_revenue_share_pct,

    DENSE_RANK() OVER (
        PARTITION BY sps.store_id
        ORDER BY
            sps.total_revenue DESC,
            sps.product_id
    ) AS revenue_rank_within_store,

    DENSE_RANK() OVER (
        PARTITION BY sps.store_id
        ORDER BY
            sps.total_quantity_sold DESC,
            sps.product_id
    ) AS quantity_rank_within_store,

    'TRUE_PROFITABILITY_NOT_CALCULABLE_NO_UNIT_COST'
        AS profitability_status

FROM store_product_sales sps

INNER JOIN store_totals st
    ON sps.store_id = st.store_id;


-- ============================================================================
-- 8. Store Product Diversity
--
-- Answers:
--   Which store has the most diverse product sales?
--
-- Metric:
--   Number of distinct products sold.
-- ============================================================================

CREATE OR REPLACE TABLE gold.store_product_diversity
USING DELTA
AS
SELECT
    s.store_id,
    s.store_name,
    s.location,

    COUNT(
        DISTINCT st.product_id
    ) AS unique_products_sold,

    COUNT(
        DISTINCT p.category
    ) AS unique_categories_sold

FROM silver.stores s

LEFT JOIN silver.sales_transactions st
    ON s.store_id = st.store_id

LEFT JOIN silver.products p
    ON st.product_id = p.product_id

GROUP BY
    s.store_id,
    s.store_name,
    s.location;


-- ============================================================================
-- 9. Sales Analysis Verification
-- ============================================================================

SELECT
    'store_sales_summary' AS table_name,
    COUNT(*) AS record_count
FROM gold.store_sales_summary

UNION ALL

SELECT
    'top_products' AS table_name,
    COUNT(*) AS record_count
FROM gold.top_products

UNION ALL

SELECT
    'store_revenue_ranking' AS table_name,
    COUNT(*) AS record_count
FROM gold.store_revenue_ranking

UNION ALL

SELECT
    'monthly_store_sales' AS table_name,
    COUNT(*) AS record_count
FROM gold.monthly_store_sales

UNION ALL

SELECT
    'product_store_sales' AS table_name,
    COUNT(*) AS record_count
FROM gold.product_store_sales

UNION ALL

SELECT
    'store_product_diversity' AS table_name,
    COUNT(*) AS record_count
FROM gold.store_product_diversity;


-- ============================================================================
-- 10. Store Revenue Results
-- ============================================================================

SELECT
    store_id,
    store_name,
    location,
    total_revenue,
    transaction_count,
    average_transaction_value,
    unique_products_sold

FROM gold.store_sales_summary

ORDER BY total_revenue DESC;


-- ============================================================================
-- 11. Top 5 Products
-- ============================================================================

SELECT
    quantity_rank,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue,
    transaction_count

FROM gold.top_products

ORDER BY quantity_rank;


-- ============================================================================
-- 12. Store Rankings
-- ============================================================================

SELECT
    store_id,
    store_name,
    total_revenue,
    transaction_count,
    revenue_rank,
    transaction_count_rank

FROM gold.store_revenue_ranking

ORDER BY
    revenue_rank,
    store_id;


-- ============================================================================
-- 13. Monthly Store Trend
-- ============================================================================

SELECT
    store_id,
    store_name,
    year_month,
    monthly_revenue,
    previous_month_revenue,
    mom_revenue_change,
    mom_revenue_growth_pct

FROM gold.monthly_store_sales

ORDER BY
    store_id,
    month_start;


-- ============================================================================
-- 14. Product Diversity
-- ============================================================================

SELECT
    store_id,
    store_name,
    location,
    unique_products_sold,
    unique_categories_sold

FROM gold.store_product_diversity

ORDER BY unique_products_sold DESC;


-- ============================================================================
-- 15. Record Sales Analysis Stage in Audit Table
-- ============================================================================

INSERT INTO audit.pipeline_runs
SELECT
    CONCAT(
        '04_sales_',
        DATE_FORMAT(
            CURRENT_TIMESTAMP(),
            'yyyyMMddHHmmss'
        )
    ) AS run_id,

    'megamart_pipeline' AS pipeline_name,

    '04_sales' AS stage_name,

    'SUCCESS' AS status,

    CURRENT_TIMESTAMP() AS start_time,

    CURRENT_TIMESTAMP() AS end_time,

    (
        SELECT COUNT(*)
        FROM silver.sales_transactions
    ) AS records_processed,

    0 AS records_failed,

    NULL AS error_message;


-- ============================================================================
-- End of Stage 04
-- ============================================================================